# Google Colab Lab Assignment - Pretrained Model & Comparative Study

**Course Name:** Generative AI  
**Lab Title:** Fresh and Rotten Fruits Classification Using CNN and Transfer Learning  
**Institution:** MIT Academy of Engineering, Alandi  
**Date of Demonstration:** 21st August 2026  
**Venue:** E222 Classroom  

**Group Members & PRN:**
* Rushikesh Hande (202401110056)
* Soham Sabane (202401110070)
* Yadnesh Tayade (202401110057)

---

## Part 1: Identified Research Paper Understanding

*   **Problem Statement:** Delayed detection of fruit spoilage leads to massive supply chain losses and health risks. An automated system is needed to classify fruits as fresh or rotten to improve quality control.
*   **Dataset Used:** Image dataset containing various classes of fresh and rotten fruits (apples, bananas, oranges).
*   **Methodology & Models:** The study utilizes custom Convolutional Neural Networks (CNNs) and compares them against pre-trained Transfer Learning architectures (such as MobileNetV2) for feature extraction.
*   **Experimental Setup:** Images were preprocessed (resized to 128x128, normalized), split into training and testing sets, and optimized using Adam.
*   **Evaluation Metrics:** The models were evaluated using Accuracy, Precision, Recall, and F1-score.
*   **Results & Key Findings:** Transfer learning models significantly outperformed custom CNNs. The paper's best configuration achieved an overall accuracy of **97.82%**, demonstrating that leveraging pre-trained ImageNet weights accelerates convergence and prevents overfitting.
*   **Relation to Implementation:** Our implementation directly mirrors the paper by building a custom CNN and a MobileNetV2 Transfer Learning model, followed by a strict comparative evaluation.
*   **What was Learned:** Building deep learning models from scratch requires immense data. Transfer learning is a highly effective methodology when computing resources or dataset sizes are limited, allowing models to reuse previously learned visual features.


**Research paper link : https://www.researchgate.net/publication/347299542_Fresh_and_Rotten_Fruits_Classification_Using_CNN_and_Transfer_Learning

**Dataset Link: https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification

## Part 2: Dataset and Preprocessing
We will download the official "Fresh and Rotten Fruits" dataset directly from Kaggle. This ensures we are working with real, high-quality images of fruit spoilage.

In [ ]:
import os

# IMPORTANT:  Kaggle API credentials here to download the dataset automatically.
os.environ['KAGGLE_USERNAME'] = "rushikesha"
os.environ['KAGGLE_KEY'] = "65a6592fd250551ecb4a9ec941d33b78"

print("Downloading real Fresh and Rotten Fruits Dataset from Kaggle...")
!pip install -q kaggle
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification

print("Unzipping dataset...")
!unzip -q -o fruits-fresh-and-rotten-for-classification.zip -d /tmp/real_fruits
print("Dataset downloaded and extracted successfully!")

### Loading and Optimizing the Dataset
We load the dataset from the extracted directories, resize the images to 128x128 to optimize training speed, and apply prefetching for better memory management.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_dir = '/tmp/real_fruits/dataset/train'
test_dir = '/tmp/real_fruits/dataset/test'

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

print("\nLoading Training and Testing Datasets...")
train_ds = image_dataset_from_directory(train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True)
test_ds = image_dataset_from_directory(test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes identified: {class_names}")

# Optimize dataset loading performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Dataset Visualization
Before training, it is crucial to verify our data. Here we visualize a batch of the actual dataset to clearly see the visual differences between the fresh and rotten fruits.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

print("Visualizing actual fruits from the training dataset...")
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontweight='bold', color='darkred', fontsize=12)
        plt.axis("off")

plt.suptitle("Fresh vs. Rotten Fruits Dataset", fontsize=18, fontweight='bold', y=0.95)
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt

# ==========================================
# 1. Build the Custom CNN Model
# ==========================================
def build_custom_cnn(input_shape=(128, 128, 3), num_classes=6):
    model = models.Sequential([
        # Input Layer
        layers.Input(shape=input_shape),

        # Data Augmentation (Prevents Overfitting)
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),

        # Normalization (0-255 px -> 0.0-1.0)
        layers.Rescaling(1./255),

        # --- Conv Block 1 ---
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Conv Block 2 ---
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Conv Block 3 ---
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Conv Block 4 ---
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- Classification Head ---
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# Instantiate and display model summary
custom_cnn = build_custom_cnn(input_shape=(128, 128, 3), num_classes=len(class_names))
custom_cnn.summary()

# ==========================================
# 2. Compile the Model
# ==========================================
custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# ==========================================
# 3. Training Callbacks & Execution
# ==========================================
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

EPOCHS = 20

print("\n--- Training Custom CNN ---")
history = custom_cnn.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

# ==========================================
# 4. Evaluation and Performance Plotting
# ==========================================
test_loss, test_acc = custom_cnn.evaluate(test_ds)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

# Plot Loss & Accuracy Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Curve
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='navy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='crimson')
axes[0].set_title('Custom CNN Accuracy Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss Curve
axes[1].plot(history.history['loss'], label='Train Loss', color='navy')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='crimson')
axes[1].set_title('Custom CNN Loss Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.show()

## Part 3: Custom CNN Implementation (Baseline)
We define a custom Convolutional Neural Network from scratch. It uses `Conv2D` layers to extract features and `MaxPooling2D` to reduce dimensionality.

In [ ]:
from tensorflow.keras import layers, models

# Define Custom CNN Architecture
cnn_model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Rescaling(1./255), # Normalization
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

### Training the Custom CNN
We train our baseline model for 5 epochs and track the time taken.

In [ ]:
import time

print("Training Custom CNN Baseline...")
start_time = time.time()
history_cnn = cnn_model.fit(train_ds, validation_data=test_ds, epochs=5)
cnn_time = time.time() - start_time
print(f"Total CNN Training Time: {cnn_time:.2f} seconds")

Training Custom CNN Baseline...
Epoch 1/5
341/341 ━━━━━━━━━━━━━━━━━━━━ 335s 964ms/step - accuracy: 0.8009 - loss: 0.5495 - val_accuracy: 0.8973 - val_loss: 0.2881
Epoch 2/5
284/341 ━━━━━━━━━━━━━━━━━━━━ 43s 770ms/step - accuracy: 0.8989 - loss: 0.2776

## Part 4: Transfer Learning Implementation
*   **Pre-trained Model Used:** MobileNetV2
*   **Why selected:** It uses inverted residual blocks, making it highly computationally efficient. It extracts complex visual features using weights pre-trained on ImageNet.
*   **Feature Extraction/Fine-tuning:** We freeze the base model (feature extraction) and only train a newly added classification head.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model

# Build Pre-trained Model (MobileNetV2)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False # Freeze base layers

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x) # Dropout to prevent overfitting
outputs = layers.Dense(num_classes, activation='softmax')(x)

tl_model = models.Model(inputs, outputs)
tl_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl_model.summary()

### Training the Transfer Learning Model
We train the MobileNetV2-based model for 5 epochs to compare against our CNN.

In [ ]:
print("Training Transfer Learning Model (MobileNetV2)...")
start_time = time.time()
history_tl = tl_model.fit(train_ds, validation_data=test_ds, epochs=5)
tl_time = time.time() - start_time
print(f"Total Transfer Learning Training Time: {tl_time:.2f} seconds")

### Feature Map Extraction & Visualization
We pass an image through the frozen base model to visualize the internal feature maps that MobileNetV2 uses to detect edges, spots, and textures of rot.

In [ ]:
print("Extracting and Visualizing Feature Maps...")
layer_names = ['block_1_expand_relu', 'block_3_expand_relu']
outputs_fm = [base_model.get_layer(name).output for name in layer_names]
feature_map_model = Model(inputs=base_model.input, outputs=outputs_fm)

for images, _ in test_ds.take(1):
    sample_image = images[0:1]
    break

feature_maps = feature_map_model.predict(tf.keras.applications.mobilenet_v2.preprocess_input(sample_image))

for layer_name, fmap in zip(layer_names, feature_maps):
    print(f"\nFeature Map Output from Layer: {layer_name}")
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    fig.suptitle(f'Filters activated in {layer_name}', fontweight='bold', color='navy')
    for i, ax in enumerate(axes.flat):
        im = ax.matshow(fmap[0, :, :, i], cmap='magma')
        ax.axis('off')
    plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
    plt.show()

## Part 5: Comparative Study & Results
We will now extract the raw predictions to compute specific evaluation metrics independently for both models.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Generate Predictions
y_true = np.concatenate([y for x, y in test_ds], axis=0)
cnn_pred = np.argmax(cnn_model.predict(test_ds), axis=1)
tl_pred = np.argmax(tl_model.predict(test_ds), axis=1)

print("Predictions successfully generated for evaluation.")

### Metric 1: Accuracy
Accuracy measures the proportion of correctly classified fruits out of all predictions.

In [ ]:
cnn_acc = accuracy_score(y_true, cnn_pred)
tl_acc = accuracy_score(y_true, tl_pred)

print(f"Custom CNN Accuracy:        {cnn_acc * 100:.2f}%")
print(f"Transfer Learning Accuracy: {tl_acc * 100:.2f}%")

### Metric 2: Precision
Precision indicates how many of the positively predicted fruits were actually correct (reducing false positives).

In [ ]:
cnn_prec = precision_score(y_true, cnn_pred, average='macro', zero_division=0)
tl_prec = precision_score(y_true, tl_pred, average='macro', zero_division=0)

print(f"Custom CNN Precision:        {cnn_prec * 100:.2f}%")
print(f"Transfer Learning Precision: {tl_prec * 100:.2f}%")

### Metric 3: Recall
Recall measures the model's ability to find all the relevant cases (e.g., ensuring no rotten fruit slips through as fresh).

In [ ]:
cnn_rec = recall_score(y_true, cnn_pred, average='macro', zero_division=0)
tl_rec = recall_score(y_true, tl_pred, average='macro', zero_division=0)

print(f"Custom CNN Recall:        {cnn_rec * 100:.2f}%")
print(f"Transfer Learning Recall: {tl_rec * 100:.2f}%")

### Metric 4: F1-Score
The F1-Score provides the harmonic mean of Precision and Recall, giving a balanced view of model performance.

In [ ]:
cnn_rec = recall_score(y_true, cnn_pred, average='macro', zero_division=0)
tl_rec = recall_score(y_true, tl_pred, average='macro', zero_division=0)

print(f"Custom CNN Recall:        {cnn_rec * 100:.2f}%")
print(f"Transfer Learning Recall: {tl_rec * 100:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the metrics and their corresponding values
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cnn_scores = [cnn_acc, cnn_prec, cnn_rec, cnn_f1]
tl_scores = [tl_acc, tl_prec, tl_rec, tl_f1]

# Convert decimals to percentages
cnn_scores_pct = [score * 100 for score in cnn_scores]
tl_scores_pct = [score * 100 for score in tl_scores]

x = np.arange(len(metrics))  # The label locations
width = 0.35  # The width of the bars

# 2. Plotting the grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, cnn_scores_pct, width, label='Custom CNN', color='indianred', edgecolor='black')
rects2 = ax.bar(x + width/2, tl_scores_pct, width, label='Transfer Learning (MobileNetV2)', color='dodgerblue', edgecolor='black')

# 3. Add styling, labels, and title
ax.set_ylabel('Scores (%)', fontweight='bold', fontsize=12)
ax.set_title('Comparative Evaluation Metrics', fontsize=18, fontweight='bold', color='midnightblue')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontweight='bold', fontsize=12)
ax.set_ylim(0, 115) # Set y-limit slightly higher to make room for labels
ax.legend(loc='lower right', fontsize=12, frameon=True, shadow=True)

# 4. Function to auto-label the bars with exact percentages
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold', fontsize=11)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

### Loss and Accuracy Curves
Visualizing the training vs. validation curves helps us identify if a model is learning properly or overfitting.

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comparative Performance: Custom CNN vs MobileNetV2', fontsize=20, fontweight='bold', color='midnightblue')

# Custom CNN Accuracy
axes[0, 0].plot(history_cnn.history['accuracy'], label='Train Acc', color='mediumseagreen', linewidth=2.5, marker='o')
axes[0, 0].plot(history_cnn.history['val_accuracy'], label='Val Acc', color='forestgreen', linewidth=2.5, marker='s', linestyle='--')
axes[0, 0].set_title('Custom CNN: Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].legend(loc='lower right')

# Custom CNN Loss
axes[0, 1].plot(history_cnn.history['loss'], label='Train Loss', color='indianred', linewidth=2.5, marker='o')
axes[0, 1].plot(history_cnn.history['val_loss'], label='Val Loss', color='darkred', linewidth=2.5, marker='s', linestyle='--')
axes[0, 1].set_title('Custom CNN: Loss', fontsize=14, fontweight='bold')
axes[0, 1].legend(loc='upper right')

# Transfer Learning Accuracy
axes[1, 0].plot(history_tl.history['accuracy'], label='Train Acc', color='dodgerblue', linewidth=2.5, marker='o')
axes[1, 0].plot(history_tl.history['val_accuracy'], label='Val Acc', color='mediumblue', linewidth=2.5, marker='s', linestyle='--')
axes[1, 0].set_title('MobileNetV2: Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].legend(loc='lower right')

# Transfer Learning Loss
axes[1, 1].plot(history_tl.history['loss'], label='Train Loss', color='tomato', linewidth=2.5, marker='o')
axes[1, 1].plot(history_tl.history['val_loss'], label='Val Loss', color='firebrick', linewidth=2.5, marker='s', linestyle='--')
axes[1, 1].set_title('MobileNetV2: Loss', fontsize=14, fontweight='bold')
axes[1, 1].legend(loc='upper right')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

### Confusion Matrices Visualization
This shows exactly where the models make mistakes (e.g., confusing a fresh apple for a rotten apple).

In [ ]:
from sklearn.metrics import confusion_matrix # Added the missing import here!

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix Comparison', fontsize=18, fontweight='bold', color='midnightblue')

# CNN Confusion Matrix
sns.heatmap(confusion_matrix(y_true, cnn_pred), annot=True, fmt='d', cmap='flare',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0],
            linewidths=1, linecolor='white', cbar_kws={'shrink': .8})
axes[0].set_title('Custom CNN', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# TL Confusion Matrix
sns.heatmap(confusion_matrix(y_true, tl_pred), annot=True, fmt='d', cmap='mako',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1],
            linewidths=1, linecolor='white', cbar_kws={'shrink': .8})
axes[1].set_title('Transfer Learning (MobileNetV2)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Final Comparative Evaluation & Conclusion

| Evaluation Parameter | Custom CNN | Transfer Learning (MobileNetV2) |
| :--- | :--- | :--- |
| **Accuracy & F1-Score** | Lower (Prone to overfitting on complex datasets) | High (Stable, robust generalization) |
| **Precision / Recall** | Variable | Consistently higher due to pre-learned filters |
| **Training Time** | Faster per epoch, but needs many epochs to converge | Slightly slower per epoch, but converges almost instantly |
| **Number of Parameters** | Low total complexity, but high trainable parameters | High total complexity, but very low trainable parameters (frozen base) |

**Advantages and Limitations:**
*   **Custom CNN:**
    *   *Advantages:* Total architectural control, lightweight model size.
    *   *Limitations:* Severely struggles to learn complex edge/texture features of rot without massive amounts of data.
*   **Transfer Learning:**
    *   *Advantages:* Exceptional accuracy; avoids overfitting by relying on robust feature extraction from ImageNet.
    *   *Limitations:* Larger deployment size; less flexible base architecture.

**Conclusion:**
The Transfer Learning approach performed significantly better. By utilizing MobileNetV2, the model bypassed the need to learn basic visual features from scratch. This allowed it to achieve high validation accuracy and F1-scores rapidly, validating the findings of the original research paper that pretrained models are vastly superior for fruit spoilage classification.

---

### **Declaration**
I confirm that the work submitted in this assignment is my own and has been completed following academic integrity guidelines.

**Signatures:**
Rushikesh Hande (202401110056)
Soham Sabane (202401110070)
Yadnesh Tayade (202401110057)